In [ ]:
!pip install contextualbandits scikit-learn

In [ ]:
import numpy as np

class SimulatedEnvironment:
    """
    A simulated environment for a contextual bandit problem.
    The expected reward is a linear function of the context.
    """
    def __init__(self, num_arms, num_features):
        """
        Initializes the environment.
        Args:
            num_arms (int): The number of actions (arms).
            num_features (int): The dimensionality of the context vector.
        """
        self.num_arms = num_arms
        self.num_features = num_features

        # Secret weight matrix W. Each row corresponds to an arm's weight vector.
        # This is the "ground truth" the bandit algorithm needs to learn.
        self._true_weights = np.random.randn(self.num_arms, self.num_features)

        print("Simulated Environment Initialized.")
        print(f"Number of arms: {self.num_arms}")
        print(f"Number of features: {self.num_features}")

    def get_context(self):
        """
        Generates a random context vector.
        Returns:
            numpy.ndarray: A context vector of shape (num_features,).
        """
        return np.random.randn(self.num_features)

    def get_reward(self, arm_index, context):
        """
        Calculates the reward for a given arm and context.
        The reward is the true expected reward plus some Gaussian noise.
        Args:
            arm_index (int): The index of the chosen arm.
            context (numpy.ndarray): The context vector.
        Returns:
            float: The stochastic reward.
        """
        # Calculate the true expected reward (dot product)
        expected_reward = context @ self._true_weights[arm_index]

        # Add Gaussian noise to make the reward stochastic
        noise = np.random.normal(0, 0.1)

        return expected_reward + noise

    def get_optimal_reward(self, context):
        """
        Calculates the reward of the best possible arm for a given context.
        Args:
            context (numpy.ndarray): The context vector.
        Returns:
            (int, float): A tuple of (optimal_arm_index, optimal_reward).
        """
        expected_rewards = [context @ self._true_weights[i] for i in range(self.num_arms)]
        optimal_arm = np.argmax(expected_rewards)
        return optimal_arm, expected_rewards[optimal_arm]

class LinUCBAgent:
    """
    A from-scratch implementation of the LinUCB algorithm.
    """
    def __init__(self, num_arms, num_features, alpha=1.0):
        """
        Initializes the LinUCB agent.
        Args:
            num_arms (int): The number of actions.
            num_features (int): The dimensionality of the context.
            alpha (float): The exploration parameter.
        """
        self.num_arms = num_arms
        self.num_features = num_features
        self.alpha = alpha

        # Initialize A and b for each arm
        # A is the covariance matrix (d x d)
        # b is the reward vector (d x 1)
        self.A = [np.identity(self.num_features) for _ in range(self.num_arms)]
        self.b = [np.zeros(self.num_features) for _ in range(self.num_arms)]

    def choose_action(self, context):
        """
        Chooses an action based on the UCB principle.
        Args:
            context (numpy.ndarray): The current context vector.
        Returns:
            int: The index of the chosen arm.
        """
        p = np.zeros(self.num_arms)
        for arm in range(self.num_arms):
            A_inv = np.linalg.inv(self.A[arm])
            theta = A_inv @ self.b[arm]  # Estimated weight vector

            # Calculate UCB
            expected_reward = theta @ context
            uncertainty_bonus = self.alpha * np.sqrt(context @ A_inv @ context)
            p[arm] = expected_reward + uncertainty_bonus

        # Choose the arm with the highest UCB
        return np.argmax(p)

    def update(self, arm_index, context, reward):
        """
        Updates the agent's parameters based on the observed reward.
        Args:
            arm_index (int): The index of the arm that was chosen.
            context (numpy.ndarray): The context for which the action was taken.
            reward (float): The observed reward.
        """
        self.A[arm_index] += np.outer(context, context)
        self.b[arm_index] += reward * context

In [ ]:
from contextualbandits.online import BootstrappedTS
from sklearn.linear_model import SGDClassifier

def create_ts_agent(num_arms):
    """
    Creates a Thompson Sampling agent using the contextualbandits library.
    Args:
        num_arms (int): The number of actions.
    Returns:
        BootstrappedTS: An initialized Thompson Sampling agent.
    """
    # Use a simple logistic regression model from scikit-learn as the base classifier
    # The bandit algorithm will manage this model internally.
    base_classifier = SGDClassifier(loss="log_loss", max_iter=1000, tol=1e-3)
    ts_agent = BootstrappedTS(base_algorithm=base_classifier, nchoices=num_arms)
    return ts_agent

In [ ]:
import matplotlib.pyplot as plt

# --- Simulation Parameters ---
NUM_ARMS = 5
NUM_FEATURES = 10
NUM_ROUNDS = 1000
ALPHA_UCB = 2.0  # Exploration parameter for LinUCB

# --- Initialization ---
env = SimulatedEnvironment(NUM_ARMS, NUM_FEATURES)
linucb_agent = LinUCBAgent(NUM_ARMS, NUM_FEATURES, alpha=ALPHA_UCB)
ts_agent = create_ts_agent(NUM_ARMS)

# --- Data Tracking ---
regret_linucb = []
regret_ts =[]
cumulative_regret_linucb = 0
cumulative_regret_ts = 0

# --- Simulation Loop ---
for t in range(NUM_ROUNDS):
    context = env.get_context()

    # Get optimal reward for regret calculation
    optimal_arm, optimal_reward = env.get_optimal_reward(context)

    # --- LinUCB Agent ---
    chosen_arm_linucb = linucb_agent.choose_action(context)
    reward_linucb = env.get_reward(chosen_arm_linucb, context)
    linucb_agent.update(chosen_arm_linucb, context, reward_linucb)

    # Calculate and store regret
    instantaneous_regret_linucb = optimal_reward - reward_linucb
    cumulative_regret_linucb += instantaneous_regret_linucb
    regret_linucb.append(cumulative_regret_linucb)

    # --- Thompson Sampling Agent ---
    # The library expects context in a 2D array
    context_2d = context.reshape(1, -1)
    chosen_arm_ts = int(ts_agent.predict(context_2d)[0])
    reward_ts = env.get_reward(chosen_arm_ts, context)

    # The library expects reward as 0 or 1 for this classifier, so we binarize it
    # This is a simplification for the example.
    binary_reward_ts = 1 if reward_ts > 0 else 0
    ts_agent.partial_fit(context_2d, np.array([chosen_arm_ts]), np.array([binary_reward_ts]))

    # Calculate and store regret
    instantaneous_regret_ts = optimal_reward - reward_ts
    cumulative_regret_ts += instantaneous_regret_ts
    regret_ts.append(cumulative_regret_ts)

    if (t + 1) % 100 == 0:
        print(f"Round {t + 1}/{NUM_ROUNDS} completed.")

# --- Visualization ---
plt.figure(figsize=(12, 6))
plt.plot(regret_linucb, label="LinUCB (from scratch)")
plt.plot(regret_ts, label="Thompson Sampling (library)")
plt.title("Cumulative Regret Over Time")
plt.xlabel("Round")
plt.ylabel("Cumulative Regret")
plt.legend()
plt.grid(True)
plt.show()